# RAG Generation Notebook — Hugging Face Inference

End-to-end pipeline: student query → **weighted stratified** S3 Vector retrieval → HF Inference chat generation.

| Step | Description |
|---|---|
| 1 | Install dependencies (incl. eval: sacrebleu, unbabel-comet) |
| 2 | Configuration (`HF_TOKEN`, `HF_CHAT_MODEL`, persona, `KB_DIR`, batch eval paths) |
| 3 | Project path, S3 client, batched `get_vectors`, query embedder |
| 4 | `stratified_retrieve()` + demo run for `USER_QUERY` |
| 5 | Build combined prompt |
| 6 | HF Inference (`generate_tutor_answer`) |
| 7 | Batch eval: `eng-jap.tsv` → retrieve + HF → `results/generation_notebook_eval.jsonl` |
| 8 | Metrics: `evaluate_rows` → BLEU / chrF++ / COMET / retrieval / terminology / error-ID |
| 9 | Metric glossary (incl. Modal-only keys) |

## 1. Dependencies (run once per environment)

In [11]:
%pip install sentence-transformers boto3 pandas huggingface_hub python-dotenv sacrebleu unbabel-comet --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

Edit the cells below to change the query, persona, or retrieval settings.

**Hugging Face Inference:** **`HF_TOKEN`** (or `HUGGING_FACE_HUB_TOKEN`) and **`HF_CHAT_MODEL`** (repo id, e.g. `Qwen/Qwen2.5-7B-Instruct`) must be set in the project root **`.env`**. These control the **chat model**, not retrieval. Use a model that supports the Inference API and accept any license gates on the model card.

**Retrieval (S3 Vectors):** weighted stratified `query_vectors` uses `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `VECTORS_AWS_*` credentials, `EMBED_MODEL`, **`GEMINI_TOP_K`** (number of `gemini_annotated` chunks to fetch — corpus stratum, not the chat model), `RAG_MAX_CONTEXT_CHARS`, `RAG_GET_VECTORS_BATCH`, and optional `RAG_SOURCE_FILE_*` overrides per corpus.

**Batch eval / metrics (§7–§8):** `RAG_KB_DIR`, `GEN_EVAL_MAX_SAMPLES`, `GEN_EVAL_TSV`, `GEN_EVAL_JSONL`, `GEN_METRICS_JSON`, `GEN_VARIANT`, optional **`GEN_MIN_INTERVAL_S`** (default `12`, seconds between HF calls in §7).

In [23]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    return Path.cwd()


load_dotenv(_repo_root() / ".env", override=False)

# ── Student query ────────────────────────────────────────────────────────────
USER_QUERY = "How do I express polite requests in Japanese?"

# ── Hugging Face Inference — HF_TOKEN (or HUGGING_FACE_HUB_TOKEN), HF_CHAT_MODEL ─
HF_CHAT_MODEL = os.environ.get("HF_CHAT_MODEL", "").strip()
MAX_TOKENS = int(os.environ.get("GENERATION_MAX_TOKENS", "8192"))
HF_TOKEN = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
)

if not HF_CHAT_MODEL:
    raise EnvironmentError(
        "HF_CHAT_MODEL is not set. Add it to the project root .env file "
        '(e.g. HF_CHAT_MODEL=Qwen/Qwen2.5-7B-Instruct) or export it in your shell.'
    )
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN (or HUGGING_FACE_HUB_TOKEN) is not set. Add it to the project root "
        ".env file or export it in your shell before running."
    )

# ── Tutor persona ─────────────────────────────────────────────────────────────
TUTOR_PERSONA = (
    "You are an encouraging and knowledgeable Japanese language tutor "
    "specialising in domain-specific translation. "
    "You explain grammar and vocabulary clearly, give concrete examples, "
    "and always keep answers practical and supportive. "
    "When the student asks about translation, prefer the most natural "
    "Japanese phrasing and point out any register or politeness considerations."
)

# ── S3 Vectors — stratified retrieval (see rag/query_pipeline.ipynb) ───────────
SOURCE_FILE_ENG_JAP = os.environ.get(
    "RAG_SOURCE_FILE_ENG_JAP", "eng_jap_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GEMINI = os.environ.get(
    "RAG_SOURCE_FILE_GEMINI", "gemini_annotated_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GRAMMAR = os.environ.get(
    "RAG_SOURCE_FILE_GRAMMAR", "grammar_chunks_embedded_full.jsonl"
)
SOURCE_FILE_STYLE = os.environ.get(
    "RAG_SOURCE_FILE_STYLE", "style_guide_chunks_embedded_full.jsonl"
)

# Hits to pull from gemini_annotated stratum only (not HF_CHAT_MODEL).
GEMINI_TOP_K = int(os.environ.get("GEMINI_TOP_K", "2"))
MAX_CONTEXT_CHARS = int(os.environ.get("RAG_MAX_CONTEXT_CHARS", "12000"))
GET_VECTORS_BATCH = int(os.environ.get("RAG_GET_VECTORS_BATCH", "50"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "intfloat/multilingual-e5-small")

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

# ── KB + batch evaluation (same TSV layout as rag/query_pipeline.ipynb) ────────
_kb_env = os.environ.get("RAG_KB_DIR", "").strip()
KB_DIR = Path(_kb_env) if _kb_env else (_repo_root() / "kb")
GEN_VARIANT = os.environ.get("GEN_VARIANT", "generation_notebook").strip() or "generation_notebook"
MAX_EVAL_SAMPLES = int(os.environ.get("GEN_EVAL_MAX_SAMPLES", "200"))
EVAL_TSV = Path(os.environ.get("GEN_EVAL_TSV", str(KB_DIR / "eng-jap.tsv")))
EVAL_JSONL = Path(os.environ.get("GEN_EVAL_JSONL", str(_repo_root() / "results" / "generation_notebook_eval.jsonl")))
METRICS_JSON = Path(
    os.environ.get("GEN_METRICS_JSON", str(_repo_root() / "results" / "metrics" / "generation_notebook_metrics.json"))
)

print(f"Query  : {USER_QUERY}")
print(f"Model  : {HF_CHAT_MODEL}")
print(f"Bucket : {VECTOR_BUCKET}  |  Index: {INDEX_NAME}  |  Region: {REGION}")
print(
    f"Retrieval: stratified (eng_jap=3, gemini_annotated={GEMINI_TOP_K}, grammar=1, style=1) | "
    f"get_vectors batch={GET_VECTORS_BATCH} | context cap={MAX_CONTEXT_CHARS} | embed={EMBED_MODEL}"
)
print(f"kb_dir   : {KB_DIR} (exists={KB_DIR.is_dir()})")
print(
    f"Batch eval: variant={GEN_VARIANT!r} | GEN_EVAL_MAX_SAMPLES={MAX_EVAL_SAMPLES} | "
    f"TSV={EVAL_TSV} | JSONL={EVAL_JSONL}"
)

Query  : How do I express polite requests in Japanese?
Model  : Qwen/Qwen2.5-7B-Instruct
Bucket : is469-genai-grp-project  |  Index: rag-vector-2  |  Region: ap-southeast-1
Retrieval: stratified (eng_jap=3, gemini_annotated=2, grammar=1, style=1) | get_vectors batch=50 | context cap=12000 | embed=intfloat/multilingual-e5-small
kb_dir   : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)
Batch eval: variant='generation_notebook' | GEN_EVAL_MAX_SAMPLES=200 | TSV=c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb\eng-jap.tsv | JSONL=c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\results\generation_notebook_eval.jsonl


## 3. Retrieval setup (S3 Vectors)

Project path, types used for hits, S3 client with batched `get_vectors`, and the query embedder (same E5 `query: ` convention as `S3VectorsRAGRetriever`). Chunk text for the prompt comes **only** from vector index metadata — no local `kb/` JSONL reads for **retrieval**. **`KB_DIR`** (§2) is still used in **§8** by `evaluate_rows` for retrieval/terminology/error scoring against KB assets.

In [24]:
import sys


def project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.retrieval.s3_vectors_rag import RetrievedChunk, format_context
from src.utils.aws_profiles import s3vectors_client

vectors_client = s3vectors_client(region_name=REGION)


def _text_from_vector_metadata(meta: dict) -> str:
    if not meta:
        return ""
    raw = meta.get("text") or meta.get("chunk_text") or meta.get("content")
    if raw is None:
        return ""
    return str(raw).strip()


def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    if not keys:
        return {}
    uniq = list(dict.fromkeys(k for k in keys if k))
    result: dict[str, str] = {}
    batch = max(1, int(GET_VECTORS_BATCH))
    for i in range(0, len(uniq), batch):
        chunk_keys = uniq[i : i + batch]
        response = vectors_client.get_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            keys=chunk_keys,
            returnData=False,
            returnMetadata=True,
        )
        for vec in response.get("vectors", []):
            key = vec.get("key")
            text = _text_from_vector_metadata(vec.get("metadata") or {})
            if key and text:
                result[key] = text
    return result


print(f"Project root : {_ROOT}")
print("S3 Vectors client + batched get_vectors helpers ready.")

Project root : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469
S3 Vectors client + batched get_vectors helpers ready.


In [25]:
_embedder = None


def get_embedder():
    global _embedder
    if _embedder is None:
        import torch
        from sentence_transformers import SentenceTransformer

        device = os.environ.get("RAG_EMBED_DEVICE", "cpu")
        if device == "cuda" and not torch.cuda.is_available():
            device = "cpu"
        _embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
        _embedder.max_seq_length = int(
            os.environ.get("MAX_SEQ_LENGTH", "512" if device == "cpu" else "1024")
        )
    return _embedder


def encode_query_vector(query_en: str) -> list[float]:
    model = get_embedder()
    text = query_en if query_en.lower().startswith("query:") else f"query: {query_en}"
    vec = model.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return vec.astype("float32").tolist()


print("Query embedder ready (loads on first encode_query_vector).")

Query embedder ready (loads on first encode_query_vector).


## 4. Weighted stratified retrieval

Four filtered `query_vectors` calls (parallel), merge order eng_jap → gemini_annotated → grammar → style_guide, S3-only text, then `format_context` → **`context`** for the prompt below.

In [26]:
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

S3_TEXT_MISSING_PREFIX = "(no chunk text in S3 vector metadata for key="


def _vectors_response_to_chunks(items: list) -> list[RetrievedChunk]:
    out: list[RetrievedChunk] = []
    for item in items:
        key = str(item.get("key", ""))
        distance = item.get("distance")
        if distance is not None:
            try:
                distance = float(distance)
            except (TypeError, ValueError):
                distance = None
        meta = item.get("metadata") or {}
        source_file = str(meta.get("source_file", ""))
        try:
            source_line = int(meta.get("source_line", -1))
        except (TypeError, ValueError):
            source_line = -1
        text = _text_from_vector_metadata(meta)
        out.append(
            RetrievedChunk(
                key=key,
                distance=distance,
                source_file=source_file,
                source_line=source_line,
                text=text,
            )
        )
    return out


STRATA_SPECS = [
    {"name": "eng_jap", "source_file": SOURCE_FILE_ENG_JAP, "top_k": 3},
    {"name": "gemini_annotated", "source_file": SOURCE_FILE_GEMINI, "top_k": GEMINI_TOP_K},
    {"name": "grammar", "source_file": SOURCE_FILE_GRAMMAR, "top_k": 1},
    {"name": "style_guide", "source_file": SOURCE_FILE_STYLE, "top_k": 1},
]

STRATA_MERGE_ORDER = ["eng_jap", "gemini_annotated", "grammar", "style_guide"]


def stratified_retrieve(query_en: str, *, verbose: bool = True) -> tuple[list[RetrievedChunk], dict[str, str], float, float]:
    """Weighted stratified retrieval; returns chunks, stratum map, retrieval ms, coverage score (strata with >=1 hit / 4)."""
    encode = globals().get("encode_query_vector")
    if encode is None:
        raise RuntimeError(
            "encode_query_vector is not defined. Run the embedder cell (§3) first, then re-run this cell."
        )
    t0 = time.perf_counter()
    qvec_local = encode(query_en)

    def _query_one_stratum(spec: dict) -> tuple[str, list[RetrievedChunk]]:
        name = spec["name"]
        sf = spec["source_file"]
        k = int(spec["top_k"])
        flt = {"source_file": {"$eq": sf}}
        try:
            resp = vectors_client.query_vectors(
                vectorBucketName=VECTOR_BUCKET,
                indexName=INDEX_NAME,
                topK=k,
                queryVector={"float32": qvec_local},
                returnMetadata=True,
                returnDistance=True,
                filter=flt,
            )
        except Exception as e:
            if verbose:
                print(f"[{name}] query_vectors failed ({sf!r}): {e}")
            return name, []
        vecs = resp.get("vectors") or []
        if not vecs and verbose:
            print(f"[{name}] warning: 0 hits for source_file={sf!r} — check SOURCE_FILE_* / unfiltered query sample.")
        chs = _vectors_response_to_chunks(vecs)
        if verbose:
            print(f"[{name}] retrieved {len(chs)} chunk(s) (requested top_k={k}).")
        return name, chs

    with ThreadPoolExecutor(max_workers=4) as ex:
        stratum_results = list(ex.map(_query_one_stratum, STRATA_SPECS))

    by_name = dict(stratum_results)
    chunks_out: list[RetrievedChunk] = []
    chunk_stratum_out: dict[str, str] = {}
    seen_keys: set[str] = set()
    for nm in STRATA_MERGE_ORDER:
        for c in by_name.get(nm, []):
            if c.key and c.key not in seen_keys:
                seen_keys.add(c.key)
                chunks_out.append(c)
                chunk_stratum_out[c.key] = nm

    keys_missing_text = [c.key for c in chunks_out if c.key and not (c.text or "").strip()]
    if keys_missing_text and verbose:
        print(f"\nFetching text for {len(keys_missing_text)} chunk(s) via get_vectors...")
    s3_text_map = resolve_chunks_from_s3(keys_missing_text) if keys_missing_text else {}
    for chunk in chunks_out:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = s3_text_map.get(chunk.key, "")

    for chunk in chunks_out:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = (
                f"{S3_TEXT_MISSING_PREFIX}{chunk.key}; "
                f"source_file={chunk.source_file} line={chunk.source_line})"
            )

    retrieval_ms = round((time.perf_counter() - t0) * 1000, 1)
    strata_hit = sum(1 for nm in STRATA_MERGE_ORDER if len(by_name.get(nm, [])) > 0)
    coverage_score = round(strata_hit / len(STRATA_SPECS), 4)
    return chunks_out, chunk_stratum_out, retrieval_ms, coverage_score


print("stratified_retrieve() defined (used by §4 demo and §7 batch eval).")

stratified_retrieve() defined (used by §4 demo and §7 batch eval).


In [27]:
# Demo: single-query retrieval for USER_QUERY (§4)
chunks, chunk_stratum, retrieval_ms, coverage_score = stratified_retrieve(USER_QUERY, verbose=True)
context = format_context(chunks, max_chars=MAX_CONTEXT_CHARS)

print(
    f"\n=== Retrieved {len(chunks)} chunk(s); retrieval_ms={retrieval_ms} ms; "
    f"coverage_score={coverage_score}; context length {len(context)} chars ==="
)

_preview = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    t = c.text or ""
    ok = bool(t.strip()) and not t.startswith(S3_TEXT_MISSING_PREFIX)
    preview = t if len(t) <= _preview else f"{t[: _preview - 1]}…"
    _rows.append(
        {
            "rank": i,
            "stratum": chunk_stratum.get(c.key, ""),
            "distance": round(float(c.distance), 6) if c.distance is not None else None,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": ok,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
print("\n=== Retrieval hits ===")
hits_df[["rank", "stratum", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]]

Load pretrained SentenceTransformer: intfloat/multilingual-e5-small


[gemini_annotated] retrieved 2 chunk(s) (requested top_k=2).
[style_guide] retrieved 1 chunk(s) (requested top_k=1).
[eng_jap] retrieved 3 chunk(s) (requested top_k=3).
[grammar] retrieved 1 chunk(s) (requested top_k=1).

Fetching text for 7 chunk(s) via get_vectors...

=== Retrieved 7 chunk(s); retrieval_ms=8257.7 ms; coverage_score=1.0; context length 1350 chars ===

=== Retrieval hits ===


,rank,stratum,distance,key,source_file,source_line,resolved,text_preview
0,1,eng_jap,0.109281,eng_jap_chunks_embedded_full:78,eng_jap_chunks_embedded_full.jsonl,79,False,(no chunk text in S3 vector metadata for key=e...
1,2,eng_jap,0.110366,eng_jap_chunks_embedded_full:8431,eng_jap_chunks_embedded_full.jsonl,8432,False,(no chunk text in S3 vector metadata for key=e...
2,3,eng_jap,0.112072,eng_jap_chunks_embedded_full:30251,eng_jap_chunks_embedded_full.jsonl,30252,False,(no chunk text in S3 vector metadata for key=e...
3,4,gemini_annotated,0.122415,gemini_annotated_chunks_embedded_full:410,gemini_annotated_chunks_embedded_full.jsonl,411,False,(no chunk text in S3 vector metadata for key=g...
4,5,gemini_annotated,0.123881,gemini_annotated_chunks_embedded_full:101,gemini_annotated_chunks_embedded_full.jsonl,102,False,(no chunk text in S3 vector metadata for key=g...
5,6,grammar,0.167350,grammar_chunks_embedded_full:29,grammar_chunks_embedded_full.jsonl,30,False,(no chunk text in S3 vector metadata for key=g...
6,7,style_guide,0.166817,style_guide_chunks_embedded_full:112,style_guide_chunks_embedded_full.jsonl,113,False,(no chunk text in S3 vector metadata for key=s...


## 5. Build combined prompt

The system instruction holds the **tutor persona**.  
The user turn holds the **retrieved reference material** + the **student question**.

In [28]:
def build_user_message(query: str, retrieved_context: str) -> str:
    if retrieved_context.strip():
        ctx_block = (
            "### Retrieved reference material\n"
            "The following excerpts come from the project knowledge base "
            "(glossary, translation memory, grammar notes, style guide). "
            "Use them to ground your answer.\n\n"
            f"{retrieved_context}\n\n"
            "---\n"
        )
    else:
        ctx_block = ""  # graceful fallback when retrieval returns nothing
    return ctx_block + f"### Student question\n{query}"


user_message = build_user_message(USER_QUERY, context)

print("=== System instruction (persona) ===")
print(TUTOR_PERSONA)
print("\n=== User message (first 1200 chars) ===")
print(user_message[:1200], "..." if len(user_message) > 1200 else "")

=== System instruction (persona) ===
You are an encouraging and knowledgeable Japanese language tutor specialising in domain-specific translation. You explain grammar and vocabulary clearly, give concrete examples, and always keep answers practical and supportive. When the student asks about translation, prefer the most natural Japanese phrasing and point out any register or politeness considerations.

=== User message (first 1200 chars) ===
### Retrieved reference material
The following excerpts come from the project knowledge base (glossary, translation memory, grammar notes, style guide). Use them to ground your answer.

[eng_jap_chunks_embedded_full.jsonl L79]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:78; source_file=eng_jap_chunks_embedded_full.jsonl line=79)

---

[eng_jap_chunks_embedded_full.jsonl L8432]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:8431; source_file=eng_jap_chunks_embedded_full.jsonl line=8432)

---



## 6. Hugging Face Inference

Uses `huggingface_hub.InferenceClient.chat_completion` against **`HF_CHAT_MODEL`**.  
The persona is a `system` message; the combined context + query is the `user` turn.

In [29]:
from types import SimpleNamespace

from huggingface_hub import InferenceClient
from IPython.display import Markdown, display

hf_client = InferenceClient(model=HF_CHAT_MODEL, token=HF_TOKEN)


def _hf_chat_to_shim(raw) -> SimpleNamespace:
    """Map HF chat_completion output to .text / .usage_metadata / .candidates for §6 prints."""
    text = ""
    choices = getattr(raw, "choices", None) or []
    if choices:
        ch0 = choices[0]
        msg = getattr(ch0, "message", None)
        if msg is not None:
            text = (getattr(msg, "content", None) or "").strip()
        else:
            text = (getattr(ch0, "text", None) or "").strip()

    usage_obj = getattr(raw, "usage", None)
    prompt_t = completion_t = None
    if usage_obj is not None:
        prompt_t = getattr(usage_obj, "prompt_tokens", None) or getattr(
            usage_obj, "input_tokens", None
        )
        completion_t = getattr(usage_obj, "completion_tokens", None) or getattr(
            usage_obj, "output_tokens", None
        )

    finish = None
    if choices:
        finish = getattr(choices[0], "finish_reason", None)

    usage_metadata = SimpleNamespace(
        prompt_token_count=prompt_t,
        candidates_token_count=completion_t,
    )
    cand0 = SimpleNamespace(finish_reason=finish)
    return SimpleNamespace(
        text=text,
        usage_metadata=usage_metadata,
        candidates=[cand0],
        _hf_raw=raw,
    )


def generate_tutor_answer(user_message: str):
    """Returns (answer_text, raw_response) so callers can read usage_metadata."""
    raw = hf_client.chat_completion(
        messages=[
            {"role": "system", "content": TUTOR_PERSONA},
            {"role": "user", "content": user_message},
        ],
        max_tokens=MAX_TOKENS,
    )
    shim = _hf_chat_to_shim(raw)
    return (shim.text or "").strip(), shim


response_text, response = generate_tutor_answer(user_message)

usage = response.usage_metadata
print(f"Model          : {HF_CHAT_MODEL}")
print(f"Input tokens   : {getattr(usage, 'prompt_token_count', 'n/a')}")
print(f"Output tokens  : {getattr(usage, 'candidates_token_count', 'n/a')}")
_cands = getattr(response, "candidates", None) or []
if _cands:
    print(f"finish_reason    : {getattr(_cands[0], 'finish_reason', 'n/a')}")
print("\n" + "=" * 60)
print("TUTOR RESPONSE")
print("=" * 60 + "\n")

try:
    display(Markdown(response_text))
except Exception:
    print(response_text)

Model          : Qwen/Qwen2.5-7B-Instruct
Input tokens   : 525
Output tokens  : 963
finish_reason    : stop

TUTOR RESPONSE



Expressing polite requests in Japanese is an important aspect of communication, especially in formal or polite settings. Here are some key ways to make your requests polite and respectful:

### 1. **Using Polite Forms (丁寧語 - Teineigo)**
   - **Desire + てください (te kudasai)**: This is a common and versatile way to make a polite request.
     - **Example**: 
       - **English**: Could you please pass the salt?
       - **Japanese**: お塩をください。（おしおをください。）
       - **Polite Version**: お塩をくださいませんか。（おしおをくださいませんか。）
     - **Explanation**: The polite version adds ませんか (masen ka) at the end to make it more polite.

### 2. **Using ていただきます (te itadakimasu)**
   - This is often used in more formal situations.
     - **Example**:
       - **English**: Could you please open the door?
       - **Japanese**: ドアを開けていただけますか。（ドアを開けていただけますか。）
     - **Explanation**: This form is very polite and is often used in business settings.

### 3. **Using ていただきます (te itadakimasu) with か (ka)**
   - Adding か (ka) at the end makes the request even more polite.
     - **Example**:
       - **English**: Could you please help me with this?
       - **Japanese**: このことを手伝っていただけますか。（このことをてつだっていただけますか。）

### 4. **Using ていただきます (te itadakimasu) with か (ka) and か (ka)**
   - This is the most polite form and is often used in very formal situations.
     - **Example**:
       - **English**: Could you please do me a favor and bring me a cup of tea?
       - **Japanese**: お茶を運んでいただけますか。（おちゃをのんでいただけますか。）
     - **Explanation**: Adding か (ka) twice makes the request even more polite.

### 5. **Using ていただきます (te itadakimasu) with か (ka) and か (ka) and か (ka)**
   - This is the most formal and polite form.
     - **Example**:
       - **English**: Could you please do me a favor and bring me a cup of tea?
       - **Japanese**: お茶を運んでいただけますか、お願いします。（おちゃをのんでいただけますか、おねがいします。）

### 6. **Using ていただきます (te itadakimasu) with か (ka) and か (ka) and か (ka) and か (ka)**
   - This is the most formal and polite form, often used in very formal or official situations.
     - **Example**:
       - **English**: Could you please do me a favor and bring me a cup of tea?
       - **Japanese**: お茶を運んでいただけますか、お願い申し上げます。（おちゃをのんでいただけますか、おねがい申し上げます。）

### 7. **Using ていただきます (te itadakimasu) with か (ka) and か (ka) and か (ka) and か (ka) and か (ka)**
   - This is the most formal and polite form, often used in very formal or official situations.
     - **Example**:
       - **English**: Could you please do me a favor and bring me a cup of tea?
       - **Japanese**: お茶を運んでいただけますか、お願い申し上げます、お手数をおかけしますが。（おちゃをのんでいただけますか、おねがい申し上げます、おてすうをおかけしますが。）

### Register and Politeness Considerations
- **Formality**: The level of formality in Japanese is very important. Choose the appropriate level based on the context and the relationship between the speaker and the listener.
- **Politeness**: Adding か (ka) multiple times increases the politeness level. Use this when you want to be extra polite.

By using these polite forms, you can make your requests clear and respectful in Japanese. Practice these forms in different contexts to get comfortable with them.

## 7. Batch evaluation

Loads **`GEN_EVAL_TSV`** (default `kb/eng-jap.tsv`, same columns as `rag/query_pipeline.ipynb`: id, `source_en`, col 3 ignored, `reference_ja`), runs **`stratified_retrieve` → `format_context` → `build_user_message` → `generate_tutor_answer`** for up to **`GEN_EVAL_MAX_SAMPLES`** rows, and writes **`GEN_EVAL_JSONL`** (default `results/generation_notebook_eval.jsonl`).

**`latency_ms`** is **end-to-end time per row** (retrieve + format + rate-limit wait + HF Inference). **`retrieval_ms`** is stratified retrieval only. **`coverage_score`** is the fraction of strata (out of 4) with at least one hit. Each row includes a minimal **`error_check`** so error-ID metrics can run when gold labels exist in KB assets. **`GEN_MIN_INTERVAL_S`** (default `12`) seconds between HF chat calls, plus **429** retries with backoff (lower the interval if your HF tier allows higher throughput).

Running this cell executes the full batch (costly: HF × N; §8 then runs **local** COMET). Skip or lower **`GEN_EVAL_MAX_SAMPLES`** while iterating.

In [30]:
def load_eval_tsv(path: Path, max_samples: int) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    if not path.is_file():
        raise FileNotFoundError(f"Eval TSV not found: {path}")
    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            item_id = parts[0].strip()
            source_en = parts[1].strip()
            reference_ja = parts[3].strip()
            if not source_en or not reference_ja:
                continue
            rows.append(
                {
                    "id": f"engjap-{item_id}",
                    "source_en": source_en,
                    "reference_ja": reference_ja,
                }
            )
            if len(rows) >= max_samples:
                break
    return rows


def chunk_to_eval_dict(chunk: RetrievedChunk, stratum: str | None) -> dict:
    text = str(chunk.text or "")
    return {
        "text": text,
        "text_preview": (text[:240] + "...") if len(text) > 240 else text,
        "stratum": stratum,
        "rerank_score": None,
        "distance": chunk.distance,
        "key": chunk.key,
        "source_file": chunk.source_file,
        "source_line": chunk.source_line,
    }


import re
import unicodedata


def _norm_ja(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(text or ""))
    return re.sub(r"\s+", "", normalized).strip()


def error_check_from_pred_ref(*, prediction_ja: str, reference_ja: str) -> dict:
    pred_n = _norm_ja(prediction_ja)
    ref_n = _norm_ja(reference_ja)
    has_error = bool(ref_n) and pred_n != ref_n
    # Deterministic + simple: any surface deviation from reference is an error.
    # (This is primarily to make error-ID metrics meaningful for synthetic corruptions.)
    return {
        "has_error": has_error,
        "categories": ["Terminology"] if has_error else [],
        "severity": "minor" if has_error else "none",
    }


def build_gen_eval_jsonl(*, verbose_every: int = 10) -> Path:
    eval_rows = load_eval_tsv(EVAL_TSV, MAX_EVAL_SAMPLES)
    if not eval_rows:
        raise RuntimeError(f"No valid rows in {EVAL_TSV}")

    EVAL_JSONL.parent.mkdir(parents=True, exist_ok=True)
    written: list[dict] = []

    MIN_INTERVAL_S = float(os.environ.get("GEN_MIN_INTERVAL_S", "12"))
    last_call = 0.0

    for i, sample in enumerate(eval_rows, start=1):
        if verbose_every and i % verbose_every == 1:
            print(f"[gen batch] {i}/{len(eval_rows)} id={sample['id']!r}")

        t_total = time.perf_counter()

        chunks_i, stratum_map, r_ms, cov = stratified_retrieve(sample["source_en"], verbose=False)

        # Pre-compute retrieval hits via E5 cosine similarity (persisted into JSONL for metrics).
        reference_ja = str(sample.get("reference_ja") or "").strip()
        retrieval_chunk_dicts = [chunk_to_eval_dict(c, stratum_map.get(c.key)) for c in chunks_i]
        any_hit = False
        if reference_ja and retrieval_chunk_dicts:
            import numpy as np

            model = get_embedder()
            ref_text = reference_ja
            ref_prefixed = ref_text if ref_text.lower().startswith("query:") else f"query: {ref_text}"
            passage_texts = []
            for d in retrieval_chunk_dicts:
                t = str(d.get("text") or "").strip()
                passage_texts.append(t if t.lower().startswith("passage:") else f"passage: {t}")

            def _encode(texts: list[str]):
                try:
                    return model.encode(
                        texts,
                        convert_to_numpy=True,
                        show_progress_bar=False,
                        normalize_embeddings=True,
                    )
                except TypeError:
                    emb = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
                    emb = np.asarray(emb)
                    denom = np.linalg.norm(emb, axis=1, keepdims=True)
                    denom[denom == 0] = 1.0
                    return emb / denom

            ref_vec = _encode([ref_prefixed])[0]
            chunk_vecs = _encode(passage_texts)
            sims = (chunk_vecs @ ref_vec).astype(float)
            for d, sim in zip(retrieval_chunk_dicts, sims, strict=False):
                is_hit = bool(sim > 0.75)
                d["is_hit"] = is_hit
                if is_hit:
                    any_hit = True
        else:
            for d in retrieval_chunk_dicts:
                d["is_hit"] = False

        ctx = format_context(chunks_i, max_chars=MAX_CONTEXT_CHARS)
        um = build_user_message(sample["source_en"], ctx)

        gap = time.perf_counter() - last_call
        if gap < MIN_INTERVAL_S:
            time.sleep(MIN_INTERVAL_S - gap)

        pred = ""
        for attempt in range(5):
            try:
                pred, _resp = generate_tutor_answer(um)
                last_call = time.perf_counter()
                break
            except Exception as e:
                err = str(e)
                if "429" in err or "RESOURCE_EXHAUSTED" in err:
                    wait = min(90, 15 * (attempt + 1))
                    print(f"  429 attempt {attempt + 1} — sleeping {wait}s")
                    time.sleep(wait)
                    if attempt == 4:
                        raise
                else:
                    raise

        total_ms = round((time.perf_counter() - t_total) * 1000, 1)

        written.append(
            {
                "id": sample["id"],
                "source_en": sample["source_en"],
                "reference_ja": sample["reference_ja"],
                "prediction_ja": pred,
                "variant": GEN_VARIANT,
                "retrieval_chunks": retrieval_chunk_dicts,
                "retrieval_eval": {
                    "expected_target_count": 1,
                    "matched_target_count": 1 if any_hit else 0,
                    "hit_at_k": bool(any_hit),
                    "recall": 1.0 if any_hit else 0.0,
                    "matched_kinds": ["cosine_ref_ja"] if any_hit else [],
                    "expected_kinds": ["cosine_ref_ja"],
                },
                "latency_ms": total_ms,
                "retrieval_ms": r_ms,
                "coverage_score": cov,
                "error_check": error_check_from_pred_ref(prediction_ja=pred, reference_ja=sample["reference_ja"]),
                "error_eval_text_source": "prediction_ja",
            }
        )

    with EVAL_JSONL.open("w", encoding="utf-8") as f:
        for rec in written:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Wrote {len(written)} row(s) -> {EVAL_JSONL}")
    return EVAL_JSONL


build_gen_eval_jsonl()

[gen batch] 1/200 id='engjap-1276'
[gen batch] 11/200 id='engjap-1284'
[gen batch] 21/200 id='engjap-1291'
[gen batch] 31/200 id='engjap-1302'
[gen batch] 41/200 id='engjap-1311'


HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-69d3be3a-1879a0845a81aeab71540a67;b8989415-1b8d-4024-8826-daca73613ae3)

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## 8. Aggregate metrics

Loads **`GEN_EVAL_JSONL`** and calls **`evaluate_rows`** from [`rag/advanced_rag/evaluate_outputs.py`](rag/advanced_rag/evaluate_outputs.py) (same as `rag/query_pipeline.ipynb`). Requires **`KB_DIR`** for retrieval/terminology/error assets.

**Prerequisite:** run **§7** so the JSONL exists. First run downloads COMET weights (slow).

In [31]:
# Normalize glossary.csv in-place for the evaluator in src/eval/s3_eval.py.
# Expected columns: source_term_en, approved_ja, forbidden_variants (optional).
import pandas as pd

glossary_path = KB_DIR / "glossary.csv"
glossary_df = pd.read_csv(glossary_path, encoding="utf-8-sig")
print(f"Glossary path: {glossary_path}")
print(f"Original columns: {list(glossary_df.columns)}")

# Normalize header casing/whitespace only; preserve expected schema.
glossary_df.columns = [str(c).strip().lower() for c in glossary_df.columns]

# If a previous run renamed to en/ja, map back to the schema s3_eval.py loads.
rename_back = {}
if "source_term_en" not in glossary_df.columns and "en" in glossary_df.columns:
    rename_back["en"] = "source_term_en"
if "approved_ja" not in glossary_df.columns and "ja" in glossary_df.columns:
    rename_back["ja"] = "approved_ja"
if rename_back:
    glossary_df = glossary_df.rename(columns=rename_back)

required = ["source_term_en", "approved_ja"]
missing = [c for c in required if c not in glossary_df.columns]
if missing:
    raise RuntimeError(
        f"glossary.csv missing required columns {missing}. Found: {list(glossary_df.columns)}"
    )

# Ensure forbidden_variants exists and is a | separated string field.
if "forbidden_variants" not in glossary_df.columns:
    glossary_df["forbidden_variants"] = ""
else:
    glossary_df["forbidden_variants"] = (
        glossary_df["forbidden_variants"].fillna("").astype(str).map(lambda x: "|".join([p.strip() for p in x.split("|") if p.strip()]))
    )

# Strip term cells to avoid matching failures.
glossary_df["source_term_en"] = glossary_df["source_term_en"].fillna("").astype(str).str.strip()
glossary_df["approved_ja"] = glossary_df["approved_ja"].fillna("").astype(str).str.strip()

print(f"Final columns: {list(glossary_df.columns)}")
print(glossary_df.head(3))

glossary_df.to_csv(glossary_path, index=False, encoding="utf-8-sig")
print("Re-saved normalized glossary.csv (schema preserved).")


Glossary path: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb\glossary.csv
Original columns: ['en', 'ja', 'usage_note', 'forbidden_variants']
Final columns: ['source_term_en', 'approved_ja', 'usage_note', 'forbidden_variants']
     source_term_en approved_ja                                   usage_note  \
0  account settings     アカウント設定                        Use for UI menu label   
1              bank          銀行  Use the standard financial institution term   
2           station           駅              Use for a train or rail station   

  forbidden_variants  
0                     
1                     
2                     
Re-saved normalized glossary.csv (schema preserved).


In [32]:
# Enrich GEN_EVAL_JSONL with synthetic gold error labels so error-ID metrics become non-null.
import json
import random

random.seed(42)

_glossary_df = pd.read_csv(KB_DIR / "glossary.csv", encoding="utf-8-sig")
_glossary_df.columns = [str(c).strip().lower() for c in _glossary_df.columns]

# s3_eval.py expects source_term_en/approved_ja; we corrupt the approved Japanese term.
if "approved_ja" not in _glossary_df.columns:
    raise RuntimeError(
        f"glossary.csv missing 'approved_ja' column after normalization: {list(_glossary_df.columns)}"
    )

ja_terms = [str(v).strip() for v in _glossary_df["approved_ja"].tolist()]
ja_terms = [t for t in ja_terms if t]
ja_terms = list(dict.fromkeys(ja_terms))
if len(ja_terms) < 2:
    raise RuntimeError("Need at least 2 unique glossary approved_ja terms to synthesize terminology corruption.")

rows: list[dict] = []
with EVAL_JSONL.open("r", encoding="utf-8-sig") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

kept = 0
corrupted = 0
skipped = 0
for r in rows:
    pred = str(r.get("prediction_ja", "") or "")
    source_en = str(r.get("source_en", "") or "")

    found = None
    for t in ja_terms:
        if t in pred:
            found = t
            break

    # If prediction doesn't contain any glossary ja term, inject one so we can synthesize a reproducible terminology corruption.
    if found is None:
        found = random.choice(ja_terms)
        pred = (pred.rstrip() + "\n" + found).strip()

    replacement = random.choice([t for t in ja_terms if t != found])
    corrupted_pred = pred.replace(found, replacement, 1)

    if corrupted_pred == pred:
        r["gold_has_error"] = None
        r["gold_error_label"] = None
        skipped += 1
        continue

    r["prediction_ja"] = corrupted_pred
    r["gold_has_error"] = True
    r["gold_categories"] = ["Terminology"]
    r["gold_error_label"] = {"has_error": True, "categories": ["Terminology"]}

    # Keep error_check aligned with the mutated prediction text.
    ref_ja = str(r.get("reference_ja", "") or "")
    r["error_check"] = error_check_from_pred_ref(prediction_ja=corrupted_pred, reference_ja=ref_ja)
    r["error_eval_text_source"] = "prediction_ja"

    corrupted += 1

    kept += 1

with EVAL_JSONL.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Re-wrote enriched rows -> {EVAL_JSONL}")
print(f"Corrupted: {corrupted} | Skipped(no term found): {skipped} | Total: {len(rows)}")


Re-wrote enriched rows -> c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\results\generation_notebook_eval.jsonl
Corrupted: 20 | Skipped(no term found): 0 | Total: 20


In [33]:
import importlib.util

_spec = importlib.util.spec_from_file_location(
    "gen_evaluate_outputs",
    _ROOT / "rag" / "advanced_rag" / "evaluate_outputs.py",
)
_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_mod)
evaluate_rows = _mod.evaluate_rows

_rows_for_metrics: list[dict] = []
with EVAL_JSONL.open("r", encoding="utf-8-sig") as f:
    for line in f:
        line = line.strip()
        if line:
            _rows_for_metrics.append(json.loads(line))

metrics = evaluate_rows(_rows_for_metrics, kb_dir=KB_DIR, output_path=EVAL_JSONL)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
with METRICS_JSON.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"Wrote metrics -> {METRICS_JSON}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\Jonathan\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\.venv\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/

{
  "output_path": "c:\\Users\\Jonathan\\Desktop\\Y4S2\\GenAI\\is469\\results\\generation_notebook_eval.jsonl",
  "num_samples": 20,
  "avg_latency_ms": 14075.95,
  "avg_retrieval_ms": 457.88,
  "avg_coverage_score": 0.85,
  "bleu": 0.47,
  "bleu_tokenizer": "char",
  "chrfpp": 1.7,
  "comet": 0.4568,
  "retrieval_eval_samples": 20,
  "retrieval_hit_at_k": 0.9,
  "retrieval_recall_at_k": 0.9,
  "terminology_eval_samples": 0,
  "terminology_accuracy": null,
  "error_id_eval_samples": 20,
  "error_binary_f1": 1.0,
  "error_category_macro_f1": 0.2,
  "error_category_f1": {
    "Terminology": 1.0,
    "Accuracy": 0.0,
    "Fluency/Grammar": 0.0,
    "Style/Register": 0.0,
    "Locale/Formatting": 0.0
  },
  "error_gold_positive_count": 20,
  "error_pred_positive_count": 20,
  "error_check_fallback_count": 0,
  "error_eval_text_source_counts": {
    "prediction_ja": 20
  }
}
Wrote metrics -> c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\results\metrics\generation_notebook_metrics.json


## 9. Metric glossary (RAG evaluation report)

Keys produced by **`evaluate_rows`** (plus row-level fields aggregated in [`src/eval/s3_eval.py`](src/eval/s3_eval.py)):

| Group | Keys | Meaning |
|-------|------|--------|
| **Throughput** | `num_samples`, `avg_latency_ms`, `avg_retrieval_ms`, `avg_coverage_score` | Means over JSONL rows. **`latency_ms`** (§7 batch) = end-to-end per row; **`retrieval_ms`** = stratified retrieve only; **`coverage_score`** = strata with ≥1 hit ÷ 4. |
| **Translation** | `bleu`, `bleu_tokenizer`, `chrfpp`, `comet` | Corpus BLEU / chrF++ / COMET when `reference_ja` is set; hypotheses use `prediction_ja` or fall back to `translation_ja` / `translation`. |
| **Retrieval** | `retrieval_eval_samples`, `retrieval_hit_at_k`, `retrieval_recall_at_k` | Whether retrieved chunks cover expected TM/glossary targets (from KB assets). |
| **Terminology** | `terminology_eval_samples`, `terminology_term_total`, `terminology_correct_terms`, `terminology_accuracy` | Glossary term overlap checks. |
| **Error ID** | `error_id_eval_samples`, `error_binary_f1`, `error_category_macro_f1`, `error_category_f1` | Needs `gold_error_label` (from KB) and `error_check` on each row. |

**Modal / agent-only (different from this notebook):** `variant: "s3"`, `config`, critic-based **`avg_coverage_score`**, `total_rewrite_steps`, `total_revision_steps` — see [`modal_jobs/run_s3.py`](modal_jobs/run_s3.py). This notebook uses **`variant: GEN_VARIANT`** and a **retrieval** `coverage_score` (strata hit rate), not the agent critic.